# Verificar GPU y recursos

In [1]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import shutil
total, used, free = shutil.disk_usage('/workspace')
print(f'Disco total: {total/1e9:.1f} GB')
print(f'Disco usado: {used/1e9:.1f} GB')
print(f'Disco libre: {free/1e9:.1f} GB')

Tue May  5 04:40:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.65.06              Driver Version: 580.65.06      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        On  |   00000000:01:00.0 Off |                  N/A |
|  0%   37C    P8             33W /  330W |       1MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Instalar dependencias


In [2]:
import subprocess

print('Instalando dependencias...')
subprocess.run(['pip', 'install', 'nnunetv2', 'nibabel', 'scipy', 'pandas', 'tqdm', 'psutil', 'nvidia-ml-py', '-q'])
print('✓ Dependencias base instaladas')

# Fix wandb + pydantic
print('Instalando wandb con fix de pydantic...')
subprocess.run([
    'pip', 'install',
    'wandb',
    'pydantic>=2.0',
    'typing_extensions>=4.12.0',
    '--upgrade', '-q'
])
print('✓ wandb instalado')

print('Instalando MedNeXt...')
subprocess.run(['pip', 'install', 'git+https://github.com/MIC-DKFZ/MedNeXt.git', '-q'])
print('✓ MedNeXt instalado')

result = subprocess.run(
    ['python', '-c', 'from nnunet_mednext import create_mednext_v1; print("✓ MedNeXt disponible")'],
    capture_output=True, text=True
)
print(result.stdout if result.returncode == 0 else f'✗ {result.stderr}')

Instalando dependencias...



[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


✓ Dependencias base instaladas
Instalando wandb con fix de pydantic...



[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


✓ wandb instalado
Instalando MedNeXt...



[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


✓ MedNeXt instalado
✓ MedNeXt disponible



#  Configurar rutas

In [1]:
import os
from pathlib import Path

WORKSPACE      = '/workspace'
NNUNET_RAW     = f'{WORKSPACE}/nnUNet_raw'
NNUNET_PREPROC = f'{WORKSPACE}/nnUNet_preprocessed'
NNUNET_RESULTS = f'{WORKSPACE}/nnUNet_results'

for d in [NNUNET_RAW, NNUNET_PREPROC, NNUNET_RESULTS]:
    os.makedirs(d, exist_ok=True)

os.environ['nnUNet_raw']          = NNUNET_RAW
os.environ['nnUNet_preprocessed'] = NNUNET_PREPROC
os.environ['nnUNet_results']      = NNUNET_RESULTS
os.environ['nnUNet_n_proc_DA']    = '16'  # workers óptimos para MedNeXt

DATASET_ID   = 507
DATASET_NAME = f'Dataset{DATASET_ID:03d}_VerSe2020'
TRAINER      = 'nnUNetTrainerMedNeXt_250epochs'
CONFIG       = '3d_lowres'
MODEL_FOLDER = 'mednext'

DRIVE_FLAG           = '--drive-shared-with-me'
DRIVE_PREPROC        = 'gdrive:preprocessed_verse_for_training'
DRIVE_RESULTS_REMOTE = 'gdrive:results_verse'

print('✓ Rutas configuradas')
print(f'  TRAINER: {TRAINER}')
print(f'  CONFIG:  {CONFIG}')
print(f'  Workers: {os.environ["nnUNet_n_proc_DA"]}')

✓ Rutas configuradas
  TRAINER: nnUNetTrainerMedNeXt_250epochs
  CONFIG:  3d_lowres
  Workers: 16


# Copiar preprocessing desde Drive

In [4]:
import subprocess
from pathlib import Path

def rclone_copy(src, dst, desc='Copiando'):
    Path(dst).mkdir(parents=True, exist_ok=True)
    print(f'{desc}...')
    result = subprocess.run([
        'rclone', 'copy',
        '--drive-shared-with-me',
        '--transfers', '8',
        '--checkers', '16',
        '--progress',
        src, dst
    ], text=True)
    if result.returncode == 0:
        print(f'✓ {desc} completado')
    else:
        print(f'✗ Error en {desc}')

def rclone_copy_file(src, dst_dir, desc=''):
    Path(dst_dir).mkdir(parents=True, exist_ok=True)
    result = subprocess.run([
        'rclone', 'copy',
        '--drive-shared-with-me',
        src, dst_dir
    ], capture_output=True, text=True)
    if result.returncode == 0:
        print(f'  ✓ {desc}')
    else:
        print(f'  ✗ Error: {desc}')

print('Restaurando preprocessing desde Drive...\n')

rclone_copy(
    f'{DRIVE_PREPROC}/nnUNet_preprocessed/{DATASET_NAME}/nnUNetPlans_3d_lowres',
    f'{NNUNET_PREPROC}/{DATASET_NAME}/nnUNetPlans_3d_lowres',
    'nnUNetPlans_3d_lowres'
)

rclone_copy(
    f'{DRIVE_PREPROC}/nnUNet_preprocessed/{DATASET_NAME}/gt_segmentations',
    f'{NNUNET_PREPROC}/{DATASET_NAME}/gt_segmentations',
    'gt_segmentations'
)

print('Copiando JSONs...')
for json_file in ['nnUNetPlans.json', 'dataset.json', 'dataset_fingerprint.json', 'splits_final.json']:
    rclone_copy_file(
        f'{DRIVE_PREPROC}/nnUNet_preprocessed/{DATASET_NAME}/{json_file}',
        f'{NNUNET_PREPROC}/{DATASET_NAME}/',
        json_file
    )

for json_file in ['dataset.json', 'splits_info.json']:
    rclone_copy_file(
        f'{DRIVE_PREPROC}/nnUNet_raw/{DATASET_NAME}/{json_file}',
        f'{NNUNET_RAW}/{DATASET_NAME}/',
        json_file
    )

n_lowres = len(list(Path(f'{NNUNET_PREPROC}/{DATASET_NAME}/nnUNetPlans_3d_lowres').glob('*')))
print(f'\n✓ nnUNetPlans_3d_lowres: {n_lowres} archivos')
print('✓ Restauración completada')

Restaurando preprocessing desde Drive...

nnUNetPlans_3d_lowres...


2026/05/05 04:41:15 NOTICE: Config file "/root/.config/rclone/rclone.conf" not found - using defaults
2026/05/05 04:41:15 Failed to create file system for "gdrive:preprocessed_verse_for_training/nnUNet_preprocessed/Dataset507_VerSe2020/nnUNetPlans_3d_lowres": didn't find section in config file


✗ Error en nnUNetPlans_3d_lowres
gt_segmentations...


2026/05/05 04:41:16 NOTICE: Config file "/root/.config/rclone/rclone.conf" not found - using defaults
2026/05/05 04:41:16 Failed to create file system for "gdrive:preprocessed_verse_for_training/nnUNet_preprocessed/Dataset507_VerSe2020/gt_segmentations": didn't find section in config file


✗ Error en gt_segmentations
Copiando JSONs...
  ✗ Error: nnUNetPlans.json
  ✗ Error: dataset.json
  ✗ Error: dataset_fingerprint.json
  ✗ Error: splits_final.json
  ✗ Error: dataset.json
  ✗ Error: splits_info.json

✓ nnUNetPlans_3d_lowres: 783 archivos
✓ Restauración completada


# Ajustar batch size

In [5]:
import json
from pathlib import Path

plans_path = Path(NNUNET_PREPROC) / DATASET_NAME / 'nnUNetPlans.json'
with open(plans_path) as f:
    plans = json.load(f)

plans['configurations']['3d_lowres']['batch_size'] = 2
with open(plans_path, 'w') as f:
    json.dump(plans, f, indent=2)

print(f'✓ batch_size 3d_lowres: {plans["configurations"]["3d_lowres"]["batch_size"]}')

✓ batch_size 3d_lowres: 2


# Instalar trainer MedNeXt 250 epochs

In [6]:
import nnunetv2
from pathlib import Path

nnunet_dir  = Path(nnunetv2.__file__).parent
trainer_dir = nnunet_dir / 'training' / 'nnUNetTrainer' / 'variants' / 'network_architecture'
trainer_dir.mkdir(parents=True, exist_ok=True)

trainer_code = '''
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer
from nnunet_mednext import create_mednext_v1
import torch
import warnings

class MedNeXtWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.encoder = model
        self.decoder = model

    def forward(self, x):
        output = self.model(x)
        if isinstance(output, (list, tuple)):
            return output
        return [output]

class nnUNetTrainerMedNeXt_250epochs(nnUNetTrainer):
    """
    MedNeXt v1 en nnU-Net v2
    250 epochs por fold
    AdamW + CosineAnnealing
    compile=False — incompatible con GroupNorm
    """
    max_num_epochs = 250
    compile = False

    def __init__(self, plans, configuration, fold, dataset_json, device=torch.device("cuda")):
        super().__init__(plans, configuration, fold, dataset_json, device)
        self.initial_lr = 1e-3
        self.num_epochs = 250
        warnings.filterwarnings("ignore", message=".*lr_scheduler.step.*")
        warnings.filterwarnings("ignore", message=".*epoch parameter.*")

    def build_network_architecture(self, architecture_class_name, arch_init_kwargs,
                                   arch_init_kwargs_req_import, num_input_channels,
                                   num_output_channels, enable_deep_supervision):
        base_model = create_mednext_v1(
            num_input_channels=num_input_channels,
            num_classes=num_output_channels,
            model_id="B",
            kernel_size=3,
            deep_supervision=enable_deep_supervision
        )
        return MedNeXtWrapper(base_model)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.network.parameters(),
            lr=self.initial_lr,
            weight_decay=3e-5
        )
        lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=self.num_epochs,
            eta_min=1e-6
        )
        return optimizer, lr_scheduler

    def set_deep_supervision_enabled(self, enabled: bool):
        pass
    
    def perform_actual_validation(self, save_probabilities: bool = False):
        """Saltar validacion final — se hara en Colab durante inferencia"""
        self.logger.log('Mean Foreground Dice', 0, self.current_epoch - 1)
        return
 
    def on_train_end(self):
        """Deshabilitar mirroring en validacion final para evitar error de flip con lista"""
        self.inference_allowed_mirroring_axes = None
        super().on_train_end()
'''

trainer_path = trainer_dir / 'nnUNetTrainerMedNeXt_250epochs.py'
trainer_path.write_text(trainer_code)

import subprocess
result = subprocess.run(
    ['python', '-c',
     'from nnunetv2.training.nnUNetTrainer.variants.network_architecture.nnUNetTrainerMedNeXt_250epochs import nnUNetTrainerMedNeXt_250epochs; print(f"✓ num_epochs={nnUNetTrainerMedNeXt_250epochs.max_num_epochs}")'],
    capture_output=True, text=True
)
print(result.stdout if result.returncode == 0 else f'✗ {result.stderr}')

✓ num_epochs=250



# Configurar wandb

In [2]:
import wandb
import os

WANDB_API_KEY = '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'
os.environ['WANDB_API_KEY'] = WANDB_API_KEY

wandb.login(key=WANDB_API_KEY)
print('✓ wandb autenticado')

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 0252751 (0252751-universidad-panamericana) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✓ wandb autenticado


# Funciones con monitor wandb

In [4]:
import subprocess
import time
import threading
import re
import wandb
import psutil
from pathlib import Path

try:
    import pynvml
    pynvml.nvmlInit()
    NVML_AVAILABLE = True
    print('✓ pynvml disponible')
except:
    try:
        import nvidia_ml_py as pynvml
        pynvml.nvmlInit()
        NVML_AVAILABLE = True
        print('✓ nvidia-ml-py disponible')
    except:
        NVML_AVAILABLE = False
        print('⚠ GPU monitoring no disponible')

def get_gpu_stats():
    if not NVML_AVAILABLE:
        return {}
    try:
        handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        mem    = pynvml.nvmlDeviceGetMemoryInfo(handle)
        util   = pynvml.nvmlDeviceGetUtilizationRates(handle)
        temp   = pynvml.nvmlDeviceGetTemperature(handle, pynvml.NVML_TEMPERATURE_GPU)
        return {
            'gpu/utilization_pct': util.gpu,
            'gpu/vram_used_gb':    mem.used / 1e9,
            'gpu/vram_total_gb':   mem.total / 1e9,
            'gpu/vram_used_pct':   mem.used / mem.total * 100,
            'gpu/temperature_c':   temp,
        }
    except:
        return {}

def get_system_stats():
    ram = psutil.virtual_memory()
    return {
        'system/ram_used_gb':  ram.used / 1e9,
        'system/ram_total_gb': ram.total / 1e9,
        'system/ram_used_pct': ram.percent,
        'system/cpu_pct':      psutil.cpu_percent(interval=1),
    }

def parse_log_metrics(log_path):
    metrics = []
    if not Path(log_path).exists():
        return metrics
    try:
        lines = Path(log_path).read_text().split('\n')
        current = {}

        for line in lines:
            line = line.strip()
            if not line:
                continue

            clean = re.sub(r'^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}[\.\d]*:\s*', '', line).strip()

            if not clean:
                continue

            m = re.search(r'^Epoch (\d+)\s*$', clean)
            if m:
                if current and 'train/loss' in current:
                    metrics.append(current)
                current = {'epoch': int(m.group(1))}
                continue

            m = re.search(r'Current learning rate: ([\d.e+-]+)', clean)
            if m and current:
                current['train/lr'] = float(m.group(1))
                continue

            m = re.search(r'^train_loss ([-\d.nan]+)', clean)
            if m and current:
                try:
                    current['train/loss'] = float(m.group(1))
                except:
                    pass
                continue

            m = re.search(r'^val_loss ([-\d.nan]+)', clean)
            if m and current:
                try:
                    current['val/loss'] = float(m.group(1))
                except:
                    pass
                continue

            m = re.search(r'Epoch time: ([\d.]+)', clean)
            if m and current:
                current['train/epoch_time_s'] = float(m.group(1))
                continue

            m = re.search(r'New best EMA pseudo Dice: ([\d.]+)', clean)
            if m and current:
                current['val/ema_pseudo_dice'] = float(m.group(1))

        if current and 'train/loss' in current:
            metrics.append(current)

        return metrics
    except:
        return []

def monitor_training(stop_flag, config, fold):
    last_epoch_logged = -1

    while not stop_flag[0]:
        log_dir = Path(NNUNET_RESULTS) / DATASET_NAME / \
                  f'{TRAINER}__nnUNetPlans__{config}' / f'fold_{fold}'

        if log_dir.exists():
            logs = sorted(log_dir.rglob('training_log*.txt'))
            best_log = None
            max_epochs = 0
            for log in logs:
                count = log.read_text().count('Epoch time:')
                if count > max_epochs:
                    max_epochs = count
                    best_log = log

            if best_log:
                epochs_data = parse_log_metrics(best_log)
                for epoch_data in epochs_data:
                    epoch_num = epoch_data.get('epoch', -1)
                    if epoch_num > last_epoch_logged:
                        log_data = {k: v for k, v in epoch_data.items()}
                        log_data.update(get_gpu_stats())
                        log_data.update(get_system_stats())
                        log_data['fold'] = fold
                        try:
                            wandb.log(log_data, step=epoch_num + fold * 250)
                        except:
                            pass
                        last_epoch_logged = epoch_num

        time.sleep(30)

def backup_fold_to_drive(config, fold):
    local = Path(NNUNET_RESULTS) / DATASET_NAME / \
            f'{TRAINER}__nnUNetPlans__{config}' / f'fold_{fold}'
    drive_dst = f'{DRIVE_RESULTS_REMOTE}/{MODEL_FOLDER}/{DATASET_NAME}/{TRAINER}__nnUNetPlans__{config}/fold_{fold}'

    if not local.exists():
        print(f'  ⚠ No existe local fold_{fold}')
        return

    print(f'  Subiendo fold_{fold} a Drive...')
    result = subprocess.run([
        'rclone', 'copy',
        '--drive-shared-with-me',
        '--transfers', '8',
        str(local), drive_dst
    ], capture_output=True, text=True)

    if result.returncode == 0:
        print(f'  ✓ Backup fold_{fold} → Drive')
    else:
        print(f'  ✗ Error backup: {result.stderr[:200]}')

def restore_checkpoints_from_drive(config):
    drive_src = f'{DRIVE_RESULTS_REMOTE}/{MODEL_FOLDER}/{DATASET_NAME}/{TRAINER}__nnUNetPlans__{config}'
    local_dst = Path(NNUNET_RESULTS) / DATASET_NAME / \
                f'{TRAINER}__nnUNetPlans__{config}'

    result = subprocess.run([
        'rclone', 'ls',
        '--drive-shared-with-me',
        drive_src
    ], capture_output=True, text=True)

    if result.returncode == 0 and result.stdout.strip():
        print(f'Restaurando checkpoints de {config} desde Drive...')
        local_dst.mkdir(parents=True, exist_ok=True)
        subprocess.run([
            'rclone', 'copy',
            '--drive-shared-with-me',
            '--transfers', '8',
            drive_src, str(local_dst)
        ], capture_output=True, text=True)
        print(f'✓ Checkpoints restaurados')
    else:
        print(f'No hay checkpoints en Drive para {config} — empezando desde cero')

def train_fold(config, fold):
    # Verificar si ya está completo LOCALMENTE
    local_checkpoint = Path(NNUNET_RESULTS) / DATASET_NAME / \
                      f'{TRAINER}__nnUNetPlans__{config}' / f'fold_{fold}' / 'checkpoint_final.pth'
    
    if local_checkpoint.exists():
        print(f'✓ Fold {fold} ya completado localmente — saltando')
        return True

    print(f'\n{"="*55}')
    print(f'  nnU-Net — {config} — fold {fold}')
    print(f'{"="*55}')

    stop_flag = [False]
    monitor_thread = threading.Thread(
        target=monitor_training,
        args=(stop_flag, config, fold),
        daemon=True
    )
    monitor_thread.start()

    cmd = [
        'nnUNetv2_train',
        str(DATASET_ID), config, str(fold),
        '-tr', TRAINER,
        '--npz',
        '--c',
    ]
    result = subprocess.run(cmd, text=True)

    stop_flag[0] = True
    monitor_thread.join(timeout=10)

    if result.returncode == 0:
        print(f'✓ Fold {fold} completado')
        backup_fold_to_drive(config, fold)
        return True
    else:
        print(f'✗ Error en fold {fold} — returncode: {result.returncode}')
        return False

print('✓ Funciones listas con monitor wandb')

✓ pynvml disponible
✓ Funciones listas con monitor wandb


# Entrenar 5 folds con wandb

In [ ]:
import wandb
import time

wandb.init(
    project='verse-spine-segmentation',
    name='mednext-3d-lowres',
    config={
        'model':       'MedNeXt',
        'config':      CONFIG,
        'trainer':     TRAINER,
        'batch_size':  2,
        'workers':     8,
        'epochs':      250,
        'folds':       5,
        'dataset':     'VerSe2019+2020',
        'optimizer':   'AdamW',
        'lr':          0.001,
        'scheduler':   'CosineAnnealing',
        'model_id':    'B',
        'kernel_size': 3
    }
)

print('✓ wandb inicializado')
print(f'  Dashboard: {wandb.run.url}')

restore_checkpoints_from_drive(CONFIG)

print(f'\nIniciando entrenamiento MedNeXt 3D lowres — 5 folds × 250 epochs\n')
resultados = []

for fold in range(5):
    start_fold = time.time()
    success = train_fold(CONFIG, fold)
    elapsed = (time.time() - start_fold) / 3600

    if success:
        resultados.append({'fold': fold, 'horas': round(elapsed, 2)})
        wandb.log({
            'fold_completado':  fold,
            'horas_fold':       round(elapsed, 2),
            'horas_acumuladas': sum(r['horas'] for r in resultados)
        })
    else:
        print(f'⚠ Fold {fold} falló — continuando con siguiente fold')
        wandb.log({'fold_fallido': fold})

wandb.finish()
print('\n✓ Entrenamiento MedNeXt completado')
for r in resultados:
    print(f'  Fold {r["fold"]}: {r["horas"]}h')

✓ wandb inicializado
  Dashboard: https://wandb.ai/0252751-universidad-panamericana/verse-spine-segmentation/runs/pvfq3hlk
No hay checkpoints en Drive para 3d_lowres — empezando desde cero

Iniciando entrenamiento MedNeXt 3D lowres — 5 folds × 250 epochs

✓ Fold 0 ya completado localmente — saltando
✓ Fold 1 ya completado localmente — saltando

  nnU-Net — 3d_lowres — fold 2

############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 

W0505 04:48:44.440000 134117787873280 torch/fx/experimental/symbolic_shapes.py:4449] [1/1] ps0 is not in var_ranges, defaulting to unknown range.
W0505 04:51:14.956000 134117787873280 torch/fx/experimental/symbolic_shapes.py:4449] [1/2] ps0 is not in var_ranges, defaulting to unknown range.
/usr/local/lib/python3.11/dist-packages/nnunetv2/training/nnUNetTrainer/nnUNetTrainer.py:1159: RuntimeWarning: invalid value encountered in scalar divide
  global_dc_per_class = [i for i in [2 * i / (2 * i + j + k) for i, j, k in zip(tp, fp, fn)]]


using pin_memory on device 0

This is the configuration used by this training:
Configuration name: 3d_lowres
 {'data_identifier': 'nnUNetPlans_3d_lowres', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 2, 'patch_size': [128, 128, 128], 'median_image_size_in_voxels': [193, 212, 195], 'spacing': [1.806111234669415, 1.7637805026068503, 1.7637805026068503], 'normalization_schemes': ['CTNormalization'], 'use_mask_for_norm': [False], 'resampling_fn_data': 'resample_data_or_seg_to_shape', 'resampling_fn_seg': 'resample_data_or_seg_to_shape', 'resampling_fn_data_kwargs': {'is_seg': False, 'order': 3, 'order_z': 0, 'force_separate_z': None}, 'resampling_fn_seg_kwargs': {'is_seg': True, 'order': 1, 'order_z': 0, 'force_separate_z': None}, 'resampling_fn_probabilities': 'resample_data_or_seg_to_shape', 'resampling_fn_probabilities_kwargs': {'is_seg': False, 'order': 1, 'order_z': 0, 'force_separate_z': None}, 'architecture': {'network_class_name': 'dynamic_network_architectures.archite

Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/nnunetv2/inference/predict_from_raw_data.py", line 683, in predict_sliding_window_return_logits
    predicted_logits = self._internal_predict_sliding_window_return_logits(data, slicers,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/_contextlib.py", line 116, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/nnunetv2/inference/predict_from_raw_data.py", line 647, in _internal_predict_sliding_window_return_logits
    raise e
  File "/usr/local/lib/python3.11/dist-packages/nnunetv2/inference/predict_from_raw_data.py", line 630, in _internal_predict_sliding_window_return_logits
    predicted_logits[sl] += prediction
RuntimeError: output with shape [28, 128, 128, 128] doesn't match the broadcast shape [1, 28, 128, 128, 128]


✗ Error en fold 2 — returncode: 1
⚠ Fold 2 falló — continuando con siguiente fold

  nnU-Net — 3d_lowres — fold 3

############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-05-05 12:26:02.720926: Using torch.compile...
2026-05-05 12:26:03.400242: do_dummy_2d_data_aug: False
2026-05-05 12:26:03.400667: Using splits from existing split file: /workspa

...

using pin_memory on device 0

This is the configuration used by this training:
Configuration name: 3d_lowres
 {'data_identifier': 'nnUNetPlans_3d_lowres', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 2, 'patch_size': [128, 128, 128], 'median_image_size_in_voxels': [193, 212, 195], 'spacing': [1.806111234669415, 1.7637805026068503, 1.7637805026068503], 'normalization_schemes': ['CTNormalization'], 'use_mask_for_norm': [False], 'resampling_fn_data': 'resample_data_or_seg_to_shape', 'resampling_fn_seg': 'resample_data_or_seg_to_shape', 'resampling_fn_data_kwargs': {'is_seg': False, 'order': 3, 'order_z': 0, 'force_separate_z': None}, 'resampling_fn_seg_kwargs': {'is_seg': True, 'order': 1, 'order_z': 0, 'force_separate_z': None}, 'resampling_fn_probabilities': 'resample_data_or_seg_to_shape', 'resampling_fn_probabilities_kwargs': {'is_seg': False, 'order': 1, 'order_z': 0, 'force_separate_z': None}, 'architecture': {'network_class_name': 'dynamic_network_architectures.archite

Traceback (most recent call last):
  File "/usr/local/bin/nnUNetv2_train", line 8, in <module>
    sys.exit(run_training_entry())
             ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/nnunetv2/run/run_training.py", line 268, in run_training_entry
    run_training(args.dataset_name_or_id, args.configuration, args.fold, args.tr, args.p, args.pretrained_weights,
  File "/usr/local/lib/python3.11/dist-packages/nnunetv2/run/run_training.py", line 213, in run_training
    nnunet_trainer.perform_actual_validation(export_validation_probabilities)
  File "/usr/local/lib/python3.11/dist-packages/nnunetv2/training/nnUNetTrainer/variants/network_architecture/nnUNetTrainerMedNeXt_250epochs.py", line 66, in perform_actual_validation
    self.logger.log('Mean Foreground Dice', 0, self.current_epoch - 1)
  File "/usr/local/lib/python3.11/dist-packages/nnunetv2/training/logging/nnunet_logger.py", line 66, in log
    self.local_logger.log(key, value, step)
  File "/usr/local/

✗ Error en fold 3 — returncode: -6
⚠ Fold 3 falló — continuando con siguiente fold

  nnU-Net — 3d_lowres — fold 4

############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-05-05 20:01:15.327909: Using torch.compile...
2026-05-05 20:01:15.942181: do_dummy_2d_data_aug: False
2026-05-05 20:01:15.942606: Using splits from existing split file: /worksp

W0505 20:01:41.678000 126068721037312 torch/fx/experimental/symbolic_shapes.py:4449] [1/1] ps0 is not in var_ranges, defaulting to unknown range.
W0505 20:03:50.976000 126068721037312 torch/fx/experimental/symbolic_shapes.py:4449] [1/2] ps0 is not in var_ranges, defaulting to unknown range.
/usr/local/lib/python3.11/dist-packages/nnunetv2/training/nnUNetTrainer/nnUNetTrainer.py:1159: RuntimeWarning: invalid value encountered in scalar divide
  global_dc_per_class = [i for i in [2 * i / (2 * i + j + k) for i, j, k in zip(tp, fp, fn)]]


using pin_memory on device 0

This is the configuration used by this training:
Configuration name: 3d_lowres
 {'data_identifier': 'nnUNetPlans_3d_lowres', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 2, 'patch_size': [128, 128, 128], 'median_image_size_in_voxels': [193, 212, 195], 'spacing': [1.806111234669415, 1.7637805026068503, 1.7637805026068503], 'normalization_schemes': ['CTNormalization'], 'use_mask_for_norm': [False], 'resampling_fn_data': 'resample_data_or_seg_to_shape', 'resampling_fn_seg': 'resample_data_or_seg_to_shape', 'resampling_fn_data_kwargs': {'is_seg': False, 'order': 3, 'order_z': 0, 'force_separate_z': None}, 'resampling_fn_seg_kwargs': {'is_seg': True, 'order': 1, 'order_z': 0, 'force_separate_z': None}, 'resampling_fn_probabilities': 'resample_data_or_seg_to_shape', 'resampling_fn_probabilities_kwargs': {'is_seg': False, 'order': 1, 'order_z': 0, 'force_separate_z': None}, 'architecture': {'network_class_name': 'dynamic_network_architectures.archite

# Hacer Run Resume de todas las Epochs

In [8]:
import wandb
import re
from pathlib import Path

# Leer todos los logs de todos los folds
wandb.init(
    project='verse-spine-segmentation',
    name='mednext-3d-lowres-completo',
    reinit=True
)

global_step = 0
for fold in range(5):
    log_dir = Path(NNUNET_RESULTS) / DATASET_NAME / \
              f'{TRAINER}__nnUNetPlans__{CONFIG}' / f'fold_{fold}'
    logs = sorted(log_dir.rglob('training_log*.txt'))
    if not logs:
        continue
    best_log = max(logs, key=lambda l: l.read_text().count('Epoch time:'))
    epochs_data = parse_log_metrics(best_log)
    
    for epoch_data in epochs_data:
        log_data = {k: v for k, v in epoch_data.items()}
        log_data['fold'] = fold
        wandb.log(log_data)
        global_step += 1

wandb.finish()
print('✓ Run resumen creado en wandb')

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


epoch,▁▃▄▅▆▁▂▂▄▄▆██▂▂▄▅▆▆▆▇▁▂▂▃▄▅▅▅▇▇█▂▃▃▅▅▆▆▇
fold,▁▁▁▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▃▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆██████
train/epoch_time_s,▃▂▂▂▂▂▂▂▂▂▂▇▂▂▆▂▂▄▂▂▂▄▂▂▂█▁▁▂▁▁█▄█▁▁▁▁▁▁
train/loss,█▇▇▇▂▂▂▁▁▁▂▂▂▁▁▁▁▃▂▂▂▁▁▁▁██▂▂▂▂▂▂▁▁▂▁▁▁▁
train/lr,█▇▆▃▃▁▁█▆▆▅▄▄▃▃▁▁▇▇▆▄▄▃▁▁▁███▇▅▄▁▁█▇▃▂▂▁
val/ema_pseudo_dice,▁▅▆▆▆▇▇▂▂▃▇▇▇██▁▃▆▇▇▇███▂▅▅▅▅▆▇▇▇█▄▅▆▇▇█
val/loss,▅▂▂▂▂▂▂▂▁▂▅▁▂▁▁▁█▆▆▂▁▂▁▁▁▇▁▁▁▁▁▁▁█▃▁▁▁▁▁
epoch,249
fold,4
train/epoch_time_s,107.28
train/loss,-0.9234


✓ Run resumen creado en wandb
